# Module 3.7 — Compressible Flow

**Everything changes when density can vary:**
Modules 1–3.6 assumed incompressible flow — density is constant, pressure propagates at infinite speed, and the N-S equations can be split into momentum and a Poisson equation. None of this holds for high-speed flows.

**When does compressibility matter?**
$$M = \frac{u}{c_s}, \qquad c_s = \sqrt{\gamma p/\rho} \approx 340 \text{ m/s in air at sea level}$$

- $M < 0.3$: density changes $< 5\%$ → treat as incompressible
- $M > 0.3$: compressibility matters — density, temperature, pressure all change
- $M > 1$: information cannot travel upstream — elliptic → hyperbolic PDE change

Applications: aircraft ($M \approx 0.8$–0.9 cruise), jet engines ($M > 1$ nozzle), rockets, re-entry vehicles, supersonic wind tunnels.

**Roadmap:**
1. The compressible Euler equations — conservation form, why it differs from incompressible
2. The equation of state — closing the system with thermodynamics
3. Normal shock relations — what happens across a shock
4. The Riemann problem — the building block of compressible FVM
5. Lax-Friedrichs and upwind Riemann solvers
6. The Sod shock tube — the standard 1D benchmark

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. The Compressible Euler Equations

### Why a new formulation is needed

In incompressible flow: pressure propagates at infinite speed → everywhere responds instantly to a perturbation → elliptic PDE → Poisson equation for pressure.

In compressible flow: pressure propagates at speed $c_s$ (finite) → hyperbolic PDE → information travels in well-defined directions (characteristics) → fundamentally different numerics.

### The system in conservation form

$$\frac{\partial \mathbf{U}}{\partial t} + \frac{\partial \mathbf{F}}{\partial x} = 0$$

where:

$$\mathbf{U} = \begin{pmatrix}\rho \\ \rho u \\ \rho E\end{pmatrix}, \qquad \mathbf{F} = \begin{pmatrix}\rho u \\ \rho u^2 + p \\ (\rho E + p)u\end{pmatrix}$$

**Three equations, three physical laws:**
- $\partial_t\rho + \partial_x(\rho u) = 0$ — mass conservation (continuity)
- $\partial_t(\rho u) + \partial_x(\rho u^2 + p) = 0$ — momentum conservation (Newton's 2nd law)
- $\partial_t(\rho E) + \partial_x[(\rho E + p)u] = 0$ — energy conservation (1st law of thermodynamics)

**The new quantity: total energy per unit volume**
$$E = e + \frac{u^2}{2}$$

where $e$ is internal energy. For an ideal gas: $e = c_v T = p/(\gamma-1)\rho$.

### Closing the system: equation of state

Four unknowns: $\rho, u, p, E$ — but only 3 equations. The **ideal gas EOS** closes it:

$$p = (\gamma-1)\rho\left(E - \frac{u^2}{2}\right) = (\gamma-1)\rho e$$

Given $\mathbf{U} = (\rho, \rho u, \rho E)$, recover:
- $\rho = U_0$
- $u = U_1/U_0$
- $E = U_2/U_0$
- $p = (\gamma-1)(U_2 - U_1^2/(2U_0))$

For air: $\gamma = 1.4$, $c_v = 717$ J/(kg·K), $c_p = 1004$ J/(kg·K).

## 2. Normal Shock Relations

Across a normal shock, the Rankine-Hugoniot conditions give the jump in state variables. For a stationary shock with upstream Mach number $M_1 > 1$:

$$\frac{\rho_2}{\rho_1} = \frac{(\gamma+1)M_1^2}{(\gamma-1)M_1^2+2}, \qquad \frac{p_2}{p_1} = \frac{2\gamma M_1^2-(\gamma-1)}{\gamma+1}, \qquad M_2^2 = \frac{(\gamma-1)M_1^2+2}{2\gamma M_1^2-(\gamma-1)}$$

Key properties:
- Density, pressure, temperature **increase** across a shock
- Velocity and Mach number **decrease** — supersonic → subsonic
- **Total pressure decreases** — entropy is produced (irreversible!)
- The stronger the shock (higher $M_1$), the greater the total pressure loss

## 3. The Riemann Problem

### The building block of compressible FVM

At each cell face, you have two states: $\mathbf{U}_L$ (left cell) and $\mathbf{U}_R$ (right cell). The **Riemann problem** asks: given this initial discontinuity, what flux should we assign to the face?

The exact solution consists of three waves: left-going shock or rarefaction, contact discontinuity, right-going shock or rarefaction. The flux is computed from the exact state at the face after time $dt$.

**Approximate Riemann solvers** (much cheaper):

| Solver | Key idea | Cost | Quality |
|--------|----------|------|--------|
| Lax-Friedrichs | Max wave speed dissipation | Very low | Diffusive but robust |
| Roe | Linearise at average state | Low | Accurate, can fail at vacuum |
| HLLC | Three-wave model | Low | Good, handles all cases |
| Exact Godunov | Full Riemann solve | High | Perfect — but too expensive |

In [ ]:
# ── Normal shock relations ────────────────────────────────────────────────────

gamma = 1.4
M1    = np.linspace(1.0, 5.0, 200)

rho_ratio = (gamma+1)*M1**2 / ((gamma-1)*M1**2 + 2)
p_ratio   = (2*gamma*M1**2 - (gamma-1)) / (gamma+1)
T_ratio   = p_ratio / rho_ratio   # ideal gas: T2/T1 = p2/p1 * rho1/rho2
M2        = np.sqrt(((gamma-1)*M1**2+2) / (2*gamma*M1**2-(gamma-1)))
# Total pressure ratio (entropy increase)
p0_ratio = (rho_ratio**(gamma/(gamma-1))) / (p_ratio**(1/(gamma-1)))

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

plots = [
    (axes[0,0], M2,        '$M_2$ (downstream Mach)', 'steelblue'),
    (axes[0,1], p_ratio,   '$p_2/p_1$ (pressure jump)',  'tomato'),
    (axes[1,0], rho_ratio, '$\\rho_2/\\rho_1$ (density jump)', 'seagreen'),
    (axes[1,1], p0_ratio,  '$p_{02}/p_{01}$ (total pressure — must decrease)', 'purple'),
]

for ax, field, label, color in plots:
    ax.plot(M1, field, color=color, lw=2.5)
    ax.set_xlabel('Upstream Mach number $M_1$', fontsize=10)
    ax.set_ylabel(label, fontsize=10)
    ax.set_title(label.split('(')[0].strip())
    ax.grid(True, alpha=0.3)
    ax.axvline(1.0, color='gray', lw=0.8, linestyle='--')
    # Annotate key values
    for M_check in [1.5, 2.0, 3.0]:
        idx = np.argmin(np.abs(M1 - M_check))
        ax.plot(M1[idx], field[idx], 'ko', ms=5)
        ax.annotate(f'{field[idx]:.2f}', xy=(M1[idx], field[idx]),
                    xytext=(M1[idx]+0.05, field[idx]*1.05), fontsize=8)

axes[1,1].axhline(1.0, color='gray', lw=0.8, linestyle='--')
axes[1,1].set_ylim(0, 1.05)

plt.suptitle(f'Normal shock relations ($\\gamma = {gamma}$, air)\n'
             'All quantities jump discontinuously at the shock', fontsize=12)
plt.tight_layout()
plt.show()

print('Normal shock at M1=2.0:')
M1_val = 2.0
print(f'  p2/p1   = {(2*gamma*M1_val**2-(gamma-1))/(gamma+1):.3f}  (pressure jumps ~{(2*gamma*M1_val**2-(gamma-1))/(gamma+1):.1f}×)')
print(f'  ρ2/ρ1   = {(gamma+1)*M1_val**2/((gamma-1)*M1_val**2+2):.3f}')
print(f'  M2      = {np.sqrt(((gamma-1)*M1_val**2+2)/(2*gamma*M1_val**2-(gamma-1))):.4f}  (always subsonic)')
print(f'  p02/p01 = {((gamma+1)*M1_val**2/((gamma-1)*M1_val**2+2))**(gamma/(gamma-1)) / ((2*gamma*M1_val**2-(gamma-1))/(gamma+1))**(1/(gamma-1)):.4f}  (entropy increase)')

In [ ]:
# ── Sod shock tube: standard 1D compressible benchmark ───────────────────────
# Left state: ρ=1, u=0, p=1
# Right state: ρ=0.125, u=0, p=0.1
# Solution at t=0.2: left rarefaction fan + contact + right shock

gamma = 1.4

def prim_to_cons(rho, u, p):
    E = p/(gamma-1)/rho + 0.5*u**2
    return np.array([rho, rho*u, rho*E])

def cons_to_prim(U):
    rho = U[0]
    u   = U[1]/rho
    E   = U[2]/rho
    p   = (gamma-1)*rho*(E - 0.5*u**2)
    return rho, u, max(p, 1e-10)

def euler_flux(U):
    rho, u, p = cons_to_prim(U)
    E = U[2]/rho
    return np.array([rho*u, rho*u**2+p, (rho*E+p)*u])

def lax_friedrichs(UL, UR, dx, dt):
    """Lax-Friedrichs numerical flux."""
    return 0.5*(euler_flux(UL) + euler_flux(UR)) - 0.5*(dx/dt)*(UR-UL)

# ── Grid ─────────────────────────────────────────────────────────────────────
N    = 300
L    = 1.0;  dx = L/N
x    = np.linspace(dx/2, L-dx/2, N)

# Initial condition
U = np.zeros((3, N))
for i, xi in enumerate(x):
    if xi < 0.5:
        U[:, i] = prim_to_cons(1.0, 0.0, 1.0)
    else:
        U[:, i] = prim_to_cons(0.125, 0.0, 0.1)

# ── Time march ────────────────────────────────────────────────────────────────
T_end = 0.2
t     = 0.0
CFL   = 0.45

while t < T_end:
    rho_arr, u_arr, p_arr = zip(*[cons_to_prim(U[:, i]) for i in range(N)])
    rho_arr = np.array(rho_arr)
    u_arr   = np.array(u_arr)
    p_arr   = np.array(p_arr)
    a_arr   = np.sqrt(np.maximum(gamma * p_arr / rho_arr, 1e-10))
    S_max   = np.max(np.abs(u_arr) + a_arr)
    dt      = CFL * dx / S_max
    dt      = min(dt, T_end - t)

    # Fluxes using Lax-Friedrichs
    F = np.zeros((3, N+1))
    F[:, 0]  = lax_friedrichs(U[:,0], U[:,0], dx, dt)     # left BC
    F[:, -1] = lax_friedrichs(U[:,-1], U[:,-1], dx, dt)   # right BC
    for i in range(1, N):
        F[:, i] = lax_friedrichs(U[:, i-1], U[:, i], dx, dt)

    U -= dt/dx * (F[:, 1:] - F[:, :-1])
    t += dt

# Extract solution
rho_f = np.array([cons_to_prim(U[:,i])[0] for i in range(N)])
u_f   = np.array([cons_to_prim(U[:,i])[1] for i in range(N)])
p_f   = np.array([cons_to_prim(U[:,i])[2] for i in range(N)])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, field, label in zip(axes, [rho_f, u_f, p_f],
                             ['Density $\\rho$', 'Velocity $u$', 'Pressure $p$']):
    ax.plot(x, field, 'b-', lw=2)
    ax.set_xlabel('x'); ax.set_ylabel(label)
    ax.set_title(label + f'\nat $t={T_end}$')
    ax.grid(True, alpha=0.3)
    # Annotate wave structures
    ax.axvspan(0.26, 0.49, alpha=0.1, color='blue', label='Rarefaction fan')
    ax.axvline(0.685, color='green', ls='--', alpha=0.7, label='Contact discontinuity')
    ax.axvline(0.850, color='red',   ls='--', alpha=0.7, label='Shock')
    ax.legend(fontsize=7)

plt.suptitle('Sod Shock Tube: Lax-Friedrichs scheme at $t=0.2$\n'
             'Three wave structures: rarefaction + contact + shock', fontsize=11)
plt.tight_layout()
plt.show()

print('The Sod tube contains three wave structures:')
print('  1. Left rarefaction fan (x≈0.26-0.49): smooth continuous expansion')
print('  2. Contact discontinuity (x≈0.685): density/entropy jump, p+u continuous')
print('  3. Right shock (x≈0.850): all variables jump — density by ~3x, p by ~3x')

## Summary

| Property | Incompressible | Compressible |
|----------|---------------|---------------|
| Density | Constant | Variable |
| Pressure propagation | Infinite speed | Speed of sound $c_s$ |
| PDE type | Elliptic (pressure) | Hyperbolic |
| Equation for $p$ | Poisson equation | Equation of state |
| Governing equations | N-S + continuity | Euler or N-S + energy |
| Shocks | ❌ No | ✅ Yes — discontinuities |
| Key parameter | Re | $M = u/c_s$ |

**Normal shock memory items:**
- Density and pressure always increase across a shock (Rankine-Hugoniot)
- Mach number always decreases — supersonic → subsonic for normal shock
- Total pressure always decreases — entropy is produced at the shock

**Sod shock tube:** the standard 1D validation for compressible solvers, equivalent to Ghia et al. for incompressible.

---
**Next:** Module 3.8 — Physics-Informed Neural Networks (PINNs): where machine learning meets CFD.